# 0420_publish_to_powerbi
Writes each Gold materialized view that has a `publish_name` to one parquet file in
`lakehouse/<env>/publish`, which the Power BI model reads through its `DataFolder` parameter.

Why a separate step: a Delta table is a folder of part files plus a transaction log, while
Power BI's parquet connector reads single files. Each Gold table is written once more as a
single parquet file (Spark keeps the exact column types), then renamed to its `publish_name`.
Only Gold flows whose latest run succeeded are published, so a failed run never overwrites
what the report shows.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

In [ ]:
import shutil


def publish_table(config):
    """Write one Gold table as a single parquet file named after its publish_name."""
    df = read_fresh(table_full_name("gold", config["destination_schema"], config["destination_table"]))
    # Lineage and change-tracking columns (_batch_id, _row_hash) stay in the lakehouse
    df = df.select([column for column in df.columns if not column.startswith("_")])
    staging = PUBLISH_FOLDER / f"_staging_{config['publish_name']}"
    df.coalesce(1).write.mode("overwrite").parquet(staging.as_posix())
    part_file = next(staging.glob("part-*.parquet"))
    target = PUBLISH_FOLDER / f"{config['publish_name']}.parquet"
    shutil.move(str(part_file), str(target))
    shutil.rmtree(staging, ignore_errors=True)
    return df.count()

In [ ]:
PUBLISH_FOLDER.mkdir(parents=True, exist_ok=True)
published, held_back = [], []

for flow in get_active_flows("GOLD"):
    if not flow["publish_name"]:
        continue
    if latest_status(flow["control_id"]) != "SUCCEEDED":
        held_back.append(flow["destination_table"])
        continue
    rows = publish_table(flow)
    published.append(flow["publish_name"])
    print(f"  {flow['publish_name'] + '.parquet':<26} {rows:>10,} rows")

print(f"Published {len(published)} file(s) to {PUBLISH_FOLDER}")
if held_back:
    raise RuntimeError(f"Not published, latest Gold run did not succeed: {held_back}")